# Multi-Label Intent Classification on Malaysian Online Forum Discussions
## Comparative Study: Logistic Regression vs. Linear Support Vector Classifier (Linear SVC)

This notebook implements an end-to-end multi-label text classification pipeline to classify Malaysian online forum posts (from Lowyat.NET / Malay-English code-switched text) into **six communicative intent categories**:
1. **Inquiry**
2. **Complaint**
3. **Opinion**
4. **Information**
5. **Expressive**
6. **Spam**

### Pipeline Overview:
- **Section 0:** Environment Setup & Configuration
- **Section 1:** Dataset Loading & Comprehensive Exploratory Data Analysis (EDA)
- **Section 2:** Rigorous 17-Step Text Preprocessing Pipeline for Noisy Multilingual Forum Text
- **Section 3:** TF-IDF Feature Engineering, Multi-label Splitting & One-vs-Rest Model Training
- **Section 4:** In-Depth Model Evaluation, Decision Boundaries, Probability Curves, and Comparative Visualizations

## 0. Environment Setup & Configuration
Install and import all necessary libraries for NLP processing, morphology, machine learning, and visualization.

In [ ]:
# Install dependencies if not already present
# !pip install -q nltk spacy malaya PySastrawi emoji contractions scikit-learn seaborn matplotlib tqdm joblib

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import html
import difflib
from IPython.display import display, Markdown, HTML

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Helper to print markdown in notebook output
def printmd(string):
    display(Markdown(string))

printmd("**Environment initialized successfully.**")

## 1. Load Dataset & Exploratory Data Analysis (EDA)
### 1.1 Load Dataset & Preview Records

In [ ]:
# Load cleaned dataset
dataset_path = 'online-forum-communicative-dataset.csv'
df = pd.read_csv(dataset_path)

target_cols = ['Inquiry', 'Complaint', 'Opinion', 'Information', 'Expressive', 'Spam']

printmd(f"**Dataset Dimensions:** `{df.shape[0]:,}` rows $\times$ `{df.shape[1]}` columns")
printmd("#### Figure 3.2: Preview of First 5 Dataset Records")
display(df.head(5))

### Figure 3.3: Complaint Labeled Post Example
Demonstrates a sample forum post labeled with the **Complaint** intent category.

In [ ]:
printmd("#### Figure 3.3: Complaint Labeled Post Example")
sample_complaint = df[(df['Complaint'] == 1) & (df['text'].str.len() > 20)][['id', 'text', 'Complaint', 'Opinion', 'Inquiry']].head(1)
display(sample_complaint)

### Figure 3.4: Pure Opinion Labeled Post Example
Demonstrates a sample forum post labeled exclusively with the **Opinion** intent category.

In [ ]:
printmd("#### Figure 3.4: Pure Opinion Labeled Post Example")
sample_opinion = df[(df['Opinion'] == 1) & (df['Complaint'] == 0) & (df['text'].str.len() > 20)][['id', 'text', 'Opinion', 'Complaint', 'Information']].head(1)
display(sample_opinion)

### Figure 3.5: Raw Forum Post Samples with Noise and Code-Switching
Showcases representative raw forum comments containing BBCode quotes, emojis, and multilingual Manglish/Malay code-switching.

In [ ]:
printmd("#### Figure 3.5: Raw Forum Post Samples with Noise and Code-Switching")
sample_quote = df[df['text'].str.contains(r'QUOTE\(', regex=True, na=False)].head(1)
sample_emoji = df[df['text'].str.contains(r'[\U00010000-\U0010ffff]', regex=True, na=False)].head(1)
sample_mixed = df[df['text'].str.contains(r'\b(padu|betul|keta|dlm|bunuh)\b', regex=True, na=False)].head(1)

noise_samples_df = pd.concat([sample_quote, sample_emoji, sample_mixed]).drop_duplicates()
display(noise_samples_df[['id', 'text']])

### 1.2 Individual Intent Distribution & Average Post Length (Word Count)
Summarizes post frequency and average word length for each of the six intent categories.

In [ ]:
# Compute word count for each post
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))

intent_stats = []
for intent in target_cols:
    subset = df[df[intent] == 1]
    intent_stats.append({
        'Intent Category': intent,
        'Total Posts': len(subset),
        'Dataset Prevalence (%)': f"{len(subset) / len(df) * 100:.2f}%",
        'Average Word Count': round(subset['word_count'].mean(), 2),
        'Median Word Count': int(subset['word_count'].median()),
        'Min Word Count': subset['word_count'].min(),
        'Max Word Count': subset['word_count'].max()
    })

intent_stats_df = pd.DataFrame(intent_stats)
printmd("#### Intent Class Distributions & Post Length Statistics:")
display(intent_stats_df)

### 1.3 Target Intent Class Distributions (Bar Charts & Tables)
Visualizes overall class frequencies across the dataset.

In [ ]:
# Plot individual intent class distribution
intent_counts = df[target_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.bar(intent_counts.index, intent_counts.values, color=sns.color_palette("Blues_r", len(intent_counts)), edgecolor='black')
plt.title("Target Intent Class Distributions", fontsize=13, fontweight='bold')
plt.xlabel("Communicative Intent Category", fontsize=11, fontweight='bold')
plt.ylabel("Number of Comments", fontsize=11, fontweight='bold')
plt.ylim(0, max(intent_counts.values) * 1.15)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + max(intent_counts.values)*0.02, f"{int(yval):,}\n({yval/len(df)*100:.1f}%)", ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.4 Multi-Label Intent Combinations (Co-occurrence Matrix & Breakdown Table)
Analyzes co-occurrence patterns between multi-label intent pairs across the forum corpus.

In [ ]:
# Multi-label co-occurrence matrix
co_occurrence = df[target_cols].T.dot(df[target_cols])

plt.figure(figsize=(8, 6.5))
sns.heatmap(co_occurrence, annot=True, fmt=',d', cmap='Blues', cbar=True, annot_kws={"size": 10, "fontweight": "bold"})
plt.title("Intent Label Co-Occurrence Matrix", fontsize=13, fontweight='bold')
plt.xlabel("Intent Category", fontsize=11, fontweight='bold')
plt.ylabel("Intent Category", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# Create intent combination signature per row
def get_intent_combination(row):
    active = [col for col in target_cols if row[col] == 1]
    return ' + '.join(active) if active else 'None'

df['intent_combination'] = df.apply(get_intent_combination, axis=1)

### 1.5 Top 10 & Complete Breakdown of Intent Combinations Including Opinion, Inquiry, and Spam

In [ ]:
# Combinations including Opinion
opinion_combinations = df[df['Opinion'] == 1]['intent_combination'].value_counts()
top_10_opinion = opinion_combinations.head(10)

plt.figure(figsize=(15, 6))
sns.barplot(x=top_10_opinion.index, y=top_10_opinion.values, palette="Blues_r")
plt.title("Top 10 Intent Combinations Including Opinion", fontsize=13, fontweight='bold')
plt.xlabel("Intent Combination", fontsize=11, fontweight='bold')
plt.ylabel("Total Post Count", fontsize=11, fontweight='bold')
plt.ylim(0, top_10_opinion.values[0] * 1.15)
plt.xticks(rotation=30, ha='right', fontsize=9.5)
for i, v in enumerate(top_10_opinion.values):
    plt.text(i, v + top_10_opinion.values[0]*0.02, f"{v:,}\n({v/len(df)*100:.2f}%)", ha='center', fontsize=8.5, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.6 Top 10 & Complete Breakdown of Intent Combinations Including Expressive, Information, and Complaint

In [ ]:
# Combinations including Expressive
expressive_combinations = df[df['Expressive'] == 1]['intent_combination'].value_counts()
top_10_expressive = expressive_combinations.head(10)

plt.figure(figsize=(15, 6))
sns.barplot(x=top_10_expressive.index, y=top_10_expressive.values, palette="Greens_r")
plt.title("Top 10 Intent Combinations Including Expressive", fontsize=13, fontweight='bold')
plt.xlabel("Intent Combination", fontsize=11, fontweight='bold')
plt.ylabel("Total Post Count", fontsize=11, fontweight='bold')
plt.ylim(0, top_10_expressive.values[0] * 1.15)
plt.xticks(rotation=30, ha='right', fontsize=9.5)
for i, v in enumerate(top_10_expressive.values):
    plt.text(i, v + top_10_expressive.values[0]*0.02, f"{v:,}\n({v/len(df)*100:.2f}%)", ha='center', fontsize=8.5, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Text Preprocessing Pipeline
### Preprocessing Change Inspection Helper (Visual Colored HTML Diff View)
We define `show_step_diff()` using Python's `difflib.SequenceMatcher` and `html.escape()` to render clean, side-by-side HTML diff views for both single-sample figure demonstrations and batch DataFrame transformations.

In [ ]:
import difflib
import html
from tqdm.auto import tqdm
tqdm.pandas()

def show_step_diff(prev_series=None, curr_series=None, step_name="", step_num=None, before_text=None, after_text=None):
    # Single text inputs (for Figure demonstrations)
    if before_text is not None and after_text is not None:
        b_words = str(before_text).split()
        a_words = str(after_text).split()
        matcher = difflib.SequenceMatcher(None, b_words, a_words)
        b_out, a_out = [], []
        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            b_chunk = html.escape(" ".join(b_words[i1:i2]))
            a_chunk = html.escape(" ".join(a_words[j1:j2]))
            if tag == 'equal':
                b_out.append(b_chunk)
                a_out.append(a_chunk)
            elif tag == 'replace':
                b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
                a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
            elif tag == 'delete':
                b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
            elif tag == 'insert':
                a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
        
        step_display_name = f"Step {step_num}: {step_name}" if step_num is not None else step_name
        table_html = f'''
        <table style="table-layout: fixed; width: 100%; border-collapse: collapse; border: 1px solid #ccc; margin-top: 8px; margin-bottom: 16px;">
            <thead>
                <tr style="background-color: #f5f5f5; border-bottom: 2px solid #ccc; text-align: left;">
                    <th style="width: 15%; padding: 8px 6px; font-size: 12px; text-align: left;">Step Name</th>
                    <th style="width: 42.5%; padding: 8px 10px; font-size: 12px; text-align: left;">BEFORE (Prior Step)</th>
                    <th style="width: 42.5%; padding: 8px 10px; font-size: 12px; text-align: left;">AFTER (Modified Step)</th>
                </tr>
            </thead>
            <tbody>
                <tr style="border-bottom: 1px solid #e0e0e0;">
                    <td style="padding: 8px 6px; font-weight: bold; vertical-align: top; width: 15%; color: #555; text-align: left; word-break: break-word;">{html.escape(step_display_name)}</td>
                    <td style="padding: 8px 10px; vertical-align: top; width: 42.5%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(b_out)}</td>
                    <td style="padding: 8px 10px; vertical-align: top; width: 42.5%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(a_out)}</td>
                </tr>
            </tbody>
        </table>
        '''
        display(HTML(table_html))
        return

    # Batch DataFrame Series Comparison
    if prev_series is not None and curr_series is not None:
        diff_mask = prev_series.astype(str).str.split() != curr_series.astype(str).str.split()
        total_affected = diff_mask.sum()
        printmd(f"**[{step_name}]** Rows updated: `{total_affected:,}` ({total_affected/len(prev_series)*100:.4f}%)")
        if total_affected == 0:
            printmd("_No rows modified in this step._")
            return
        changed_df = pd.DataFrame({
            "before": prev_series[diff_mask],
            "after": curr_series[diff_mask]
        })
        short_samples = changed_df[changed_df["before"].astype(str).str.len() < 250]
        remaining_samples = changed_df[~changed_df.index.isin(short_samples.index)]
        sample_df = pd.concat([short_samples, remaining_samples]).head(5) if len(changed_df) >= 5 else changed_df
        html_rows = []
        for idx, row in sample_df.iterrows():
            b_words = str(row["before"]).split()
            a_words = str(row["after"]).split()
            matcher = difflib.SequenceMatcher(None, b_words, a_words)
            b_out, a_out = [], []
            for tag, i1, i2, j1, j2 in matcher.get_opcodes():
                b_chunk = html.escape(" ".join(b_words[i1:i2]))
                a_chunk = html.escape(" ".join(a_words[j1:j2]))
                if tag == 'equal':
                    b_out.append(b_chunk)
                    a_out.append(a_chunk)
                elif tag == 'replace':
                    b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
                    a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
                elif tag == 'delete':
                    b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
                elif tag == 'insert':
                    a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
            html_rows.append(f'''
            <tr style="border-bottom: 1px solid #e0e0e0;">
                <td style="padding: 8px 6px; font-weight: bold; vertical-align: top; width: 8%; color: #555; text-align: left; word-break: break-word;">Row {idx}</td>
                <td style="padding: 8px 10px; vertical-align: top; width: 46%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(b_out)}</td>
                <td style="padding: 8px 10px; vertical-align: top; width: 46%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(a_out)}</td>
            </tr>
            ''')
        table_html = f'''
        <table style="table-layout: fixed; width: 100%; border-collapse: collapse; border: 1px solid #ccc; margin-top: 8px; margin-bottom: 16px;">
            <thead>
                <tr style="background-color: #f5f5f5; border-bottom: 2px solid #ccc; text-align: left;">
                    <th style="width: 8%; padding: 8px 6px; font-size: 12px; text-align: left;">Row ID</th>
                    <th style="width: 46%; padding: 8px 10px; font-size: 12px; text-align: left;">BEFORE (Prior Step)</th>
                    <th style="width: 46%; padding: 8px 10px; font-size: 12px; text-align: left;">AFTER (Modified Step)</th>
                </tr>
            </thead>
            <tbody>
                {''.join(html_rows)}
            </tbody>
        </table>
        '''
        display(HTML(table_html))

# Initialize clean_text column
df['clean_text'] = df['text'].copy()

# Step 1: Remove Lowyat Quotes, BBCode, Signatures, and HTML
Forum-specific noise such as nested quote blocks, spoilers, images, BBCode tags, Lowyat edit signatures, and strict HTML markup are stripped one by one.

### 1.1 Remove Quote Blocks & Quote Headers
Quotes contain text originally written by other users, which distorts the current user's intent. Both `[quote]...[/quote]` blocks and `QUOTE(...)` headers are stripped.

In [ ]:
def remove_quote_blocks(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\[quote(?:=[^\]]*)?\][\s\S]*?\[/quote\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?quote(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?quote\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'QUOTE\s*\([^\)]*?\)', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 1.1
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(remove_quote_blocks)
show_step_diff(prev_step, df['clean_text'], "Step 1.1: Remove Quote Blocks & Headers")

### Figure 3.6: Preprocessing Step 1 Removal of Forum Platform Markup
Demonstrates the removal of forum quote tags and markup headers.

In [ ]:
printmd("#### Figure 3.6: Preprocessing Step 1 Removal of Forum Platform Markup")
raw_sample_step1 = df[df['text'].str.contains(r'QUOTE\(', regex=True, na=False)]['text'].iloc[0]
cleaned_sample_step1 = remove_quote_blocks(raw_sample_step1)
show_step_diff(
    step_num=1,
    step_name="Removal of Forum Platform Markup",
    before_text=raw_sample_step1,
    after_text=cleaned_sample_step1
)

### 1.2 Remove Spoilers, Images, Media, Formatting, Signatures, and HTML
Strips additional forum BBCode elements (spoilers, images, code blocks, videos, formatting tags, edit footers) and decodes HTML entities.

In [ ]:
def remove_spoilers(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\[spoiler(?:=[^\]]*)?\][\s\S]*?\[/spoiler\]|\[/?spoiler(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)

def remove_images_and_code(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\[img(?:=[^\]]*)?\][\s\S]*?\[/img\]|\[/?img(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[code(?:=[^\]]*)?\][\s\S]*?\[/code\]|\[/?code(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    return text

def remove_media_and_attachments(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\[attachmentid=[^\]]*\]|\[attachment=[^\]]*\]|\[/attachment\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[youtube(?:=[^\]]*)?\][\s\S]*?\[/youtube\]|\[/?youtube(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[video(?:=[^\]]*)?\][\s\S]*?\[/video\]|\[/?video(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[url=[^\]]*\]|\[/url\]|\[url\]', ' ', text, flags=re.IGNORECASE)
    return text

def remove_bbcode_formatting(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\[/?(?:b|i|u|s|color|size|font|align|list|\*|center|left|right|sub|sup|table|tr|td|th)(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)

def remove_signatures_and_footers(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'--+\s*\n[\s\S]*$', ' ', text)
    text = re.sub(r'This post has been edited by[\s\S]*?(?:\n|$)', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Sent from my [^\n]+', ' ', text, flags=re.IGNORECASE)
    return text

def strip_html_tags(text):
    if not isinstance(text, str): return ""
    return re.sub(r'<[^>]+>', ' ', text)

def decode_html_entities(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    return re.sub(r'&(?:amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);', ' ', text)

# Composite forum markup removal alias
def remove_forum_markup(text):
    text = remove_quote_blocks(text)
    text = remove_spoilers(text)
    text = remove_images_and_code(text)
    text = remove_media_and_attachments(text)
    text = remove_bbcode_formatting(text)
    text = remove_signatures_and_footers(text)
    text = strip_html_tags(text)
    text = decode_html_entities(text)
    return text

# Apply remaining Step 1 functions
df['clean_text'] = df['clean_text'].progress_apply(remove_spoilers)
df['clean_text'] = df['clean_text'].progress_apply(remove_images_and_code)
df['clean_text'] = df['clean_text'].progress_apply(remove_media_and_attachments)
df['clean_text'] = df['clean_text'].progress_apply(remove_bbcode_formatting)
df['clean_text'] = df['clean_text'].progress_apply(remove_signatures_and_footers)
df['clean_text'] = df['clean_text'].progress_apply(strip_html_tags)
df['clean_text'] = df['clean_text'].progress_apply(decode_html_entities)
printmd("**Step 1 Completed:** Forum markup, BBCode, and HTML stripped.")

# Step 2: Entity Masking with Standardized Placeholder Tokens
Masks variable personal entities (Emails, URLs, Phone numbers, Malaysian NRIC numbers, Prices, Times, and Dates) with clean token tags.

In [ ]:
def mask_emails(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,7}\b', ' EMAILTOKEN ', text)

def mask_urls(text):
    if not isinstance(text, str): return ""
    return re.sub(r'https?://(?:www\.)?[\w.-]+(?:\.[a-zA-Z]{2,})+(?:/[^\s]*)?|www\.[\w.-]+(?:\.[a-zA-Z]{2,})+(?:/[^\s]*)?', ' URLTOKEN ', text)

def mask_phone_numbers(text):
    if not isinstance(text, str): return ""
    return re.sub(r'(?:\+?60|0)[1-9][0-9]?-?[0-9]{7,8}\b', ' PHONETOKEN ', text)

def mask_nric(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\b\d{6}-\d{2}-\d{4}\b|\b\d{12}\b', ' NRICTOKEN ', text)

def mask_prices(text):
    if not isinstance(text, str): return ""
    return re.sub(r'(?:\bRM|\$|MYR)\s*\d+(?:[.,]\d{2})?\b|\b\d+(?:[.,]\d{2})?\s*(?:ringgit|cents?|bucks?)\b', ' PRICETOKEN ', text, flags=re.IGNORECASE)

def mask_times(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\b(?:1[0-2]|0?[1-9]):[0-5][0-9]\s*(?:am|pm|AM|PM)\b|\b(?:[01]?[0-9]|2[0-3]):[0-5][0-9]\b', ' TIMETOKEN ', text)

def mask_dates(text):
    if not isinstance(text, str): return ""
    date_pattern = r'\b(?:\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}|\d{4}[-/.]\d{1,2}[-/.]\d{1,2}|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2},?\s+\d{4}|\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4})\b'
    return re.sub(date_pattern, ' DATETOKEN ', text, flags=re.IGNORECASE)

def mask_entities(text):
    text = mask_emails(text)
    text = mask_urls(text)
    text = mask_phone_numbers(text)
    text = mask_nric(text)
    text = mask_prices(text)
    text = mask_times(text)
    text = mask_dates(text)
    return text

# Apply Step 2 entity masking
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(mask_entities)
show_step_diff(prev_step, df['clean_text'], "Step 2: Entity Masking")

### Figure 3.7: Preprocessing Step 2 Masking of Variable Entities
Demonstrates masking of emails, URLs, phone numbers, prices, times, and dates.

In [ ]:
printmd("#### Figure 3.7: Preprocessing Step 2 Masking of Variable Entities")
raw_sample_step2 = df[df['text'].str.contains(r'https?://|RM\s*\d+|\d{1,2}:\d{2}', regex=True, na=False)]['text'].iloc[0]
cleaned_sample_step2 = mask_entities(raw_sample_step2)
show_step_diff(
    step_num=2,
    step_name="Masking of Variable Entities",
    before_text=raw_sample_step2,
    after_text=cleaned_sample_step2
)

# Step 3: Normalization of Character Elongation & Laughter Patterns
Compresses elongated repeated letters (e.g., `noooo` $\to$ `no`, `yeaaah` $\to$ `yeah`) and normalizes informal Malaysian laughter patterns (`hahaha`, `wkwk`, `kekeke`, `lmao`) to standard tokens.

In [ ]:
def compress_elongations(text):
    if not isinstance(text, str): return ""
    return re.sub(r'([a-zA-Z])\1{2,}', r'\1\1', text)

def standardize_laughter(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\b(?:ha|he|hi|ho|ah|ja|wk|kw|xa){2,}\b', ' laugh ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(?:lol|lmao|rofl|kek|kekw|lulz?)\b', ' laugh ', text, flags=re.IGNORECASE)
    return text

def normalize_elongation_and_laughter(text):
    return standardize_laughter(compress_elongations(text))

# Apply Step 3
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(normalize_elongation_and_laughter)
show_step_diff(prev_step, df['clean_text'], "Step 3: Normalize Elongations & Laughter")

### Figure 3.8: Preprocessing Step 3 Normalisation of Elongated Words and Laughter
Demonstrates character elongation compression and laughter pattern normalization.

In [ ]:
printmd("#### Figure 3.8: Preprocessing Step 3 Normalisation of Elongated Words and Laughter")
raw_sample_step3 = df[df['text'].str.contains(r'(.)\1{2,}|(ha){2,}|(wakak){1,}', regex=True, na=False)]['text'].iloc[0]
cleaned_sample_step3 = normalize_elongation_and_laughter(raw_sample_step3)
show_step_diff(
    step_num=3,
    step_name="Normalisation of Elongated Words and Laughter",
    before_text=raw_sample_step3,
    after_text=cleaned_sample_step3
)

# Step 4: Convert Emojis into Semantic Text Descriptors
Converts visual Unicode emojis (e.g. 😡, 😂, 😭) into semantic textual descriptors using the `emoji` package.

In [ ]:
import emoji

def convert_emojis(text):
    if not isinstance(text, str): return ""
    demojized = emoji.demojize(text, delimiters=(" ", " "))
    return demojized.replace("_", " ")

def convert_emojis_to_text(text):
    return convert_emojis(text)

# Apply Step 4
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(convert_emojis)
show_step_diff(prev_step, df['clean_text'], "Step 4: Convert Emojis to Text")

### Figure 3.9: Preprocessing Step 4 Conversion of Emojis to Text
Demonstrates converting Unicode emojis into semantic textual descriptors.

In [ ]:
printmd("#### Figure 3.9: Preprocessing Step 4 Conversion of Emojis to Text")
raw_sample_step4 = df[df['text'].str.contains(r'[\U00010000-\U0010ffff]', regex=True, na=False)]['text'].iloc[0]
cleaned_sample_step4 = convert_emojis(raw_sample_step4)
show_step_diff(
    step_num=4,
    step_name="Conversion of Emojis to Text",
    before_text=raw_sample_step4,
    after_text=cleaned_sample_step4
)

# Step 5: Text Cleaning, Case Normalization & Spacing
Converts all text to lowercase and collapses redundant whitespace.

In [ ]:
def normalize_spacing_and_case(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\s+', ' ', text.lower()).strip()

# Apply Step 5
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(normalize_spacing_and_case)
show_step_diff(prev_step, df['clean_text'], "Step 5: Case & Spacing Normalization")

# Step 6: Hybrid Slang Normalization (Malay, English & Custom Dictionaries)
Maps noisy Malaysian forum slang, Manglish contractions, and informal abbreviations (e.g., `xde`, `tak`, `bodo`, `dgn`, `pls`, `korg`) to their standard lexical roots using a bilingual slang dictionary.

In [ ]:
# Load Malaya dictionary or comprehensive custom slang dictionary
try:
    from malaya.dictionary import is_malay
except ImportError:
    def is_malay(w): return False

SLANG_MAP = {
    'xde': 'tidak ada', 'takde': 'tidak ada', 'tadak': 'tidak ada', 'x': 'tidak', 'tak': 'tidak', 'tk': 'tidak',
    'dgn': 'dengan', 'ngan': 'dengan', 'u': 'you', 'ur': 'your', 'pls': 'please', 'plz': 'please',
    'aq': 'aku', 'korg': 'kamu orang', 'bodo': 'bodoh', 'giler': 'gila', 'sgt': 'sangat',
    'dh': 'sudah', 'dah': 'sudah', 'tgh': 'tengah', 'sbb': 'sebab', 'sbbnye': 'sebabnya',
    'camne': 'bagaimana', 'cmne': 'bagaimana', 'mcm': 'macam', 'knp': 'kenapa', 'bila': 'bila',
    'tatau': 'tidak tahu', 'xtau': 'tidak tahu', 'ktard': 'user', 'ktards': 'users',
    'lowyat': 'forum', 'luno': 'crypto', 'binance': 'crypto', 'lazada': 'store', 'shopee': 'store'
}

def normalize_slang_hybrid(text):
    if not isinstance(text, str): return ""
    words = text.split()
    normalized = [SLANG_MAP.get(w, w) for w in words]
    return ' '.join(normalized)

def normalize_slang(text):
    return normalize_slang_hybrid(text)

# Apply Step 6
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(normalize_slang_hybrid)
show_step_diff(prev_step, df['clean_text'], "Step 6: Hybrid Slang Normalization")

### Figure 3.10: Preprocessing Step 6 Slang Normalisation via Bilingual Dictionary
Demonstrates slang and informal abbreviation normalization using the bilingual dictionary.

In [ ]:
printmd("#### Figure 3.10: Preprocessing Step 6 Slang Normalisation via Bilingual Dictionary")
raw_sample_step6 = df[df['clean_text'].str.contains(r'\b(tidak|kamu|aku|sangat|sebab)\b', regex=True, na=False)]['clean_text'].iloc[0]
cleaned_sample_step6 = normalize_slang_hybrid(raw_sample_step6)
show_step_diff(
    step_num=6,
    step_name="Slang Normalisation via Bilingual Dictionary",
    before_text=raw_sample_step6,
    after_text=cleaned_sample_step6
)

# Step 7: Expand Grammatical Contractions
Expands formal and informal English grammatical contractions (e.g., `can't` $\to$ `cannot`, `don't` $\to$ `do not`, `i'm` $\to$ `i am`).

In [ ]:
import contractions

def fix_contractions(text):
    if not isinstance(text, str): return ""
    return contractions.fix(text)

def expand_contractions(text):
    return fix_contractions(text)

# Apply Step 7
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(fix_contractions)
show_step_diff(prev_step, df['clean_text'], "Step 7: Expand Contractions")

### Figure 3.11: Preprocessing Step 7 Expansion of Contractions
Demonstrates expanding grammatical contractions.

In [ ]:
printmd("#### Figure 3.11: Preprocessing Step 7 Expansion of Contractions")
raw_sample_step7 = "I can't believe it's not working, you shouldn't have done that!"
cleaned_sample_step7 = fix_contractions(raw_sample_step7)
show_step_diff(
    step_num=7,
    step_name="Expansion of Contractions",
    before_text=raw_sample_step7,
    after_text=cleaned_sample_step7
)

# Step 8: Remove All Punctuations
All punctuation marks are stripped and replaced with spaces, ensuring zero punctuation noise survives into downstream models.

In [ ]:
import string

def remove_punctuations(text):
    if not isinstance(text, str): return ""
    punct_pattern = f"[{re.escape(string.punctuation)}]"
    return re.sub(punct_pattern, ' ', text)

# Apply Step 8
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(remove_punctuations)
show_step_diff(prev_step, df['clean_text'], "Step 8: Remove All Punctuations")

# Step 9: Split Alphanumeric Tokens & Remove Standalone Numbers
Splits letter-digit boundaries (e.g., `BUADDAY3` $\to$ `buadday 3`, `voucherMY2` $\to$ `vouchermy 2`) and eliminates standalone digits while preserving all entity masking tokens (`DATETOKEN`, `TIMETOKEN`, `PRICETOKEN`, `URLTOKEN`, `PHONETOKEN`, `NRICTOKEN`, `EMAILTOKEN`).

In [ ]:
def split_and_remove_numbers(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'(?<=[a-zA-Z])(?=\d)|(?<=\d)(?=[a-zA-Z])', ' ', text)
    text = re.sub(r'\b\d+\b', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def split_alphanumeric_and_remove_numbers(text):
    return split_and_remove_numbers(text)

# Apply Step 9
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(split_and_remove_numbers)
show_step_diff(prev_step, df['clean_text'], "Step 9: Split Alphanumerics & Remove Numbers")

### Figure 3.12: Preprocessing Step 9 Splitting Glued Alphanumeric and Number Removal
Demonstrates splitting glued alphanumeric tokens and removing standalone numbers.

In [ ]:
printmd("#### Figure 3.12: Preprocessing Step 9 Splitting Glued Alphanumeric and Number Removal")
raw_sample_step9 = "iPhone12 sold for RM2500 in 2021 at 5pm with voucherMY2 and 3rd discount"
cleaned_sample_step9 = split_and_remove_numbers(raw_sample_step9)
show_step_diff(
    step_num=9,
    step_name="Splitting Glued Alphanumeric and Number Removal",
    before_text=raw_sample_step9,
    after_text=cleaned_sample_step9
)

# Step 10: Remove Non-Latin Words / Characters
Strips non-ASCII character sequences to focus the corpus on Latin-scripted text.

In [ ]:
def remove_non_latin(text):
    if not isinstance(text, str): return ""
    return re.sub(r'[^\x00-\x7F]+', ' ', text)

# Apply Step 10
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].progress_apply(remove_non_latin)
show_step_diff(prev_step, df['clean_text'], "Step 10: Remove Non-Latin Characters")

# Step 11: Word Tokenization
Splits cleaned text into individual word tokens using NLTK `word_tokenize`.

In [ ]:
from nltk import download as nltk_download
from nltk.tokenize import word_tokenize

nltk_download('punkt', quiet=True)
nltk_download('punkt_tab', quiet=True)

def tokenize_words(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return word_tokenize(text)

# Apply Step 11
df['tokens'] = df['clean_text'].progress_apply(tokenize_words)
printmd("**Step 11 Completed:** Tokenized cleaned text into token lists.")

### Figure 3.13: Preprocessing Step 11 Word Tokenisation
Demonstrates splitting preprocessed text strings into token arrays.

In [ ]:
printmd("#### Figure 3.13: Preprocessing Step 11 Word Tokenisation")
raw_sample_step11 = df['clean_text'].iloc[4]
tokenized_step11 = tokenize_words(raw_sample_step11)
show_step_diff(
    step_num=11,
    step_name="Word Tokenisation",
    before_text=raw_sample_step11,
    after_text=str(tokenized_step11)
)

# Step 12: Language Classification and Token Tagging
Tokens are classified using **Sastrawi Malay Lexicon** for Bahasa Melayu and **NLTK WordNet / Corpus** for English, tagging tokens as `TAG`, `MALAY`, `ENGLISH`, or `UNKNOWN` to guide downstream language-specific morphological processing.

In [ ]:
from nltk.corpus import words, wordnet
import nltk
nltk.download('words', quiet=True)
nltk.download('wordnet', quiet=True)

NLTK_WORDS_SET = {w.lower() for w in words.words()}
MASKING_TAGS = {'emailtoken', 'urltoken', 'phonetoken', 'nrictoken', 'pricetoken', 'timetoken', 'datetoken'}

def is_english_nltk(w):
    return w in NLTK_WORDS_SET or len(wordnet.synsets(w)) > 0

def tag_tokens(tokens):
    tagged = []
    for token in tokens:
        token_lower = token.lower()
        if token_lower in MASKING_TAGS:
            tagged.append((token, "TAG"))
        elif is_english_nltk(token_lower):
            tagged.append((token, "ENGLISH"))
        elif is_malay(token_lower):
            tagged.append((token, "MALAY"))
        else:
            tagged.append((token, "UNKNOWN"))
    return tagged

def tag_language_and_entities(tokens):
    return tag_tokens(tokens)

# Apply Step 12
df['tagged_tokens'] = df['tokens'].progress_apply(tag_tokens)
printmd("**Step 12 Completed:** Tagged language and entity categories on tokens.")

### Figure 3.14: Preprocessing Step 12 Language and Entity Tagging Breakdown
Displays sample tokens tagged with their detected language and entity classifications.

In [ ]:
printmd("#### Figure 3.14: Preprocessing Step 12 Language and Entity Tagging Breakdown")
sample_tokens_step12 = df['tokens'].iloc[4]
tagged_step12 = tag_tokens(sample_tokens_step12)
df_tagged_breakdown = pd.DataFrame(tagged_step12, columns=["Token", "Classification Tag"])
display(df_tagged_breakdown.head(10).T)

# Step 13: Part-of-Speech (POS) Tagging for English Tokens
NLTK's `pos_tag` assigns contextual grammatical categories (`NOUN`, `VERB`, `ADJ`, `ADV`) to English tokens to guide downstream lemmatization.

In [ ]:
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
from nltk import pos_tag

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def pos_tag_english_tokens(tagged_tokens):
    token_texts = [t[0] for t in tagged_tokens]
    nltk_tags = pos_tag(token_texts)
    pos_assigned = []
    for i, (tok, lang) in enumerate(tagged_tokens):
        if lang == "ENGLISH":
            wn_pos = get_wordnet_pos(nltk_tags[i][1])
            pos_assigned.append((tok, lang, wn_pos))
        else:
            pos_assigned.append((tok, lang, None))
    return pos_assigned

# Apply Step 13
df['pos_tagged_tokens'] = df['tagged_tokens'].progress_apply(pos_tag_english_tokens)
printmd("**Step 13 Completed:** Assigned contextual POS tags to English tokens.")

# Step 14: Morphological Normalization (Lemmatization with POS & Malay Stemming)
Applies **WordNet POS Lemmatization** to English words and **Sastrawi Stemmer** to Malay words, standardizing inflected and derived words to base lemmas.

In [ ]:
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

lemmatizer = WordNetLemmatizer()
stemmer_factory = StemmerFactory()
malay_stemmer = stemmer_factory.create_stemmer()

def lemmatize_and_stem(pos_tagged_tokens):
    morph_tokens = []
    for tok, lang, pos in pos_tagged_tokens:
        tok_lower = tok.lower()
        if lang == "ENGLISH" and pos:
            lemma = lemmatizer.lemmatize(tok_lower, pos=pos)
            morph_tokens.append((lemma, lang))
        elif lang == "MALAY":
            stemmed = malay_stemmer.stem(tok_lower)
            morph_tokens.append((stemmed if stemmed else tok_lower, lang))
        else:
            morph_tokens.append((tok_lower, lang))
    return morph_tokens

# Apply Step 14
df['morph_tokens'] = df['pos_tagged_tokens'].progress_apply(lemmatize_and_stem)
printmd("**Step 14 Completed:** Morphological lemmatization and stemming applied.")

### Figure 3.15: Preprocessing Step 14 English Lemmatization vs Malay Stemming
Comparison table demonstrating language-specific morphological processing.

In [ ]:
printmd("#### Figure 3.15: Preprocessing Step 14 English Lemmatization vs Malay Stemming")
comparison_data = [
    {"Token": "playing", "Language Tag": "ENGLISH", "POS": "VERB", "Processed Output": "play", "Morphology Method": "WordNet Lemmatizer"},
    {"Token": "better", "Language Tag": "ENGLISH", "POS": "ADJ", "Processed Output": "good", "Morphology Method": "WordNet Lemmatizer"},
    {"Token": "memperbaiki", "Language Tag": "MALAY", "POS": "-", "Processed Output": "baik", "Morphology Method": "Sastrawi Stemmer"},
    {"Token": "makanan", "Language Tag": "MALAY", "POS": "-", "Processed Output": "makan", "Morphology Method": "Sastrawi Stemmer"},
    {"Token": "URLTOKEN", "Language Tag": "TAG", "POS": "-", "Processed Output": "URLTOKEN", "Morphology Method": "Protected Entity Tag"}
]
display(pd.DataFrame(comparison_data))

# Step 15: Language-Aware Stopword Removal with Protected Inquiry Interrogatives
Filters English (SpaCy) and Malay (Sastrawi) stopwords while explicitly protecting key interrogatives (`why`, `how`, `what`, `which`, `where`, `who`, `kenapa`, `mengapa`, `bagaimana`) to retain strong feature signals for Inquiry intent classification.

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOPWORDS
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

PRESERVED_WORDS = {'why', 'how', 'what', 'which', 'where', 'who', 'kenapa', 'mengapa', 'bagaimana'}
ENGLISH_STOPWORDS = set(SPACY_STOPWORDS) - PRESERVED_WORDS
sastrawi_stop_factory = StopWordRemoverFactory()
MALAY_STOPWORDS = set(sastrawi_stop_factory.get_stop_words()) - PRESERVED_WORDS

def remove_stopwords(morph_tokens):
    cleaned = []
    for word, tag in morph_tokens:
        word_lower = word.lower()
        if word_lower in ENGLISH_STOPWORDS or word_lower in MALAY_STOPWORDS:
            continue
        if word.isdigit():
            continue
        if len(word) > 1 or word_lower in MASKING_TAGS:
            cleaned.append((word, tag))
    return cleaned

def remove_stopwords_with_exceptions(tokens):
    return remove_stopwords(tokens)

# Apply Step 15
df['filtered_tokens'] = df['morph_tokens'].progress_apply(remove_stopwords)
df['clean_text'] = df['filtered_tokens'].apply(lambda t_list: ' '.join([t[0] for t in t_list]))
printmd("**Step 15 Completed:** Filtered stopwords while preserving interrogatives.")

### Figure 3.16: Preprocessing Step 15 Stopword Removal with Protected Interrogatives
Demonstrates stopword filtering with protected inquiry question words.

In [ ]:
printmd("#### Figure 3.16: Preprocessing Step 15 Stopword Removal with Protected Interrogatives")
sample_before_step15 = [('why', 'ENGLISH'), ('is', 'ENGLISH'), ('the', 'ENGLISH'), ('service', 'ENGLISH'), ('so', 'ENGLISH'), ('lousy', 'ENGLISH'), ('kenapa', 'MALAY'), ('ini', 'MALAY'), ('berlaku', 'MALAY')]
sample_after_step15 = remove_stopwords(sample_before_step15)
show_step_diff(
    step_num=15,
    step_name="Stopword Removal with Protected Interrogatives",
    before_text=str([t[0] for t in sample_before_step15]),
    after_text=str([t[0] for t in sample_after_step15])
)

# Step 16: Filter Out Empty & 0-Token Posts
Posts that contained exclusively quotes, spoilers, stripped media, or stopwords are reduced to 0 tokens. Filtering them prevents zero-vector noise from degrading TF-IDF feature matrices and classifier decision boundaries.

In [ ]:
initial_total_rows = len(df)
df = df[df['clean_text'].astype(str).str.strip().str.len() > 0].reset_index(drop=True)
dropped_rows = initial_total_rows - len(df)
printmd(f"**Step 16 Completed:** Dropped `{dropped_rows:,}` empty/0-token posts ({dropped_rows/initial_total_rows*100:.4f}% of total).")
printmd(f"**Cleaned Dataset for Model Training:** `{len(df):,}` valid informative posts remaining.")

# Step 17: Train / Test Dataset Splitting
The preprocessed corpus is partitioned into a **70% training set** and a **30% testing set**.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df[target_cols], test_size=0.30, random_state=42
)

printmd(f"**Dataset Partition:** Training set: `{len(X_train):,}` samples ({len(X_train)/len(df)*100:.1f}%), Testing set: `{len(X_test):,}` samples ({len(X_test)/len(df)*100:.1f}%)")

### Figure 3.18: Dataset Train and Test Split Architecture
Visual diagram showing the 70:30 train-test partition of the preprocessed corpus.

In [ ]:
printmd("#### Figure 3.18: Dataset Train and Test Split Architecture")
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.barh(["Dataset Split"], [70], color="#1976d2", edgecolor="black", label="Training Set (70% - 68,944 posts)")
ax.barh(["Dataset Split"], [30], left=[70], color="#ff9800", edgecolor="black", label="Testing Set (30% - 29,548 posts)")
ax.set_xlim(0, 100)
ax.set_xlabel("Percentage (%)", fontsize=11, fontweight="bold")
ax.set_title("Dataset Train-Test Split Partition (70:30)", fontsize=13, fontweight="bold")
ax.text(35, 0, "Training Set: 70%\n(68,944 samples)", ha="center", va="center", color="white", fontsize=11, fontweight="bold")
ax.text(85, 0, "Testing Set: 30%\n(29,548 samples)", ha="center", va="center", color="white", fontsize=11, fontweight="bold")
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.3), ncol=2)
plt.tight_layout()
plt.show()

# Step 18: TF-IDF Feature Vectorization
Converts cleaned text into numerical n-gram feature vectors weighted by Term Frequency-Inverse Document Frequency (TF-IDF).

In [ ]:
from joblib import dump as joblib_dump, load as joblib_load
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words=None, min_df=3, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Persist TF-IDF vectorizer artifact
joblib_dump(tfidf, "tfidf_vectorizer.joblib")
feature_names = tfidf.get_feature_names_out()
printmd(f"**TF-IDF Vocabulary Space:** Fitted `{len(feature_names):,}` unique n-gram features across `{X_train_tfidf.shape[0]:,}` training documents.")

# Extract sample top weighted terms per document for Figure 3.17
sample_tfidf_array = X_train_tfidf[:5].toarray()
top_terms_summary = []
for i in range(5):
    row_vec = sample_tfidf_array[i]
    top_word_indices = np.argsort(row_vec)[-5:][::-1]
    top_word_indices = [idx for idx in top_word_indices if row_vec[idx] > 0]
    top_keywords = ", ".join([f"{feature_names[idx]} ({row_vec[idx]:.3f})" for idx in top_word_indices])
    top_terms_summary.append({
        "Document ID": f"Doc {i} (Row {X_train.index[i]})",
        "Cleaned Text Snippet": X_train.iloc[i][:70] + ("..." if len(X_train.iloc[i]) > 70 else ""),
        "Highest TF-IDF Feature Weights": top_keywords
    })

### Figure 3.17: Sample Document TF-IDF Feature Weights
Displays the highest weighted n-gram keywords for sample preprocessed documents.

In [ ]:
printmd("#### Figure 3.17: Sample Document TF-IDF Feature Weights")
display(pd.DataFrame(top_terms_summary).head(5))

# Step 19: Train One-vs-Rest Logistic Regression Model

### Figure 3.19: Log Loss (Binary Cross-Entropy) Formula for Multi-Label Logistic Regression
$$\mathcal{L}_{\text{BCE}}(y_k, \hat{y}_k) = -\frac{1}{N}\sum_{i=1}^N \left[ y_k^{(i)} \log(\hat{y}_k^{(i)}) + (1 - y_k^{(i)}) \log(1 - \hat{y}_k^{(i)}) \right]$$

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

lr_model = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
lr_model.fit(X_train_tfidf, y_train)
lr_models = lr_model.estimators_

# Persist Logistic Regression model artifact
joblib_dump(lr_model, "logistic_regression_model.joblib")
printmd("**Logistic Regression Model:** Successfully trained and saved to `logistic_regression_model.joblib`.")

# Step 20: Logistic Regression Feature Interpretation & Sigmoid Curves
### 20.1 Coefficient Evaluation: Top & Bottom Predictive Features per Intent Class (Logistic Regression)

In [ ]:
top_words_lr = {}
bottom_words_lr = {}

for i, label in enumerate(target_cols):
    coefs = lr_model.estimators_[i].coef_[0]
    top_indices = np.argsort(coefs)[-15:][::-1]
    bottom_indices = np.argsort(coefs)[:15]
    top_words_lr[label] = [f"{feature_names[idx]} ({coefs[idx]:.2f})" for idx in top_indices]
    bottom_words_lr[label] = [f"{feature_names[idx]} ({coefs[idx]:.2f})" for idx in bottom_indices]

df_top_words_lr = pd.DataFrame(top_words_lr)
df_bottom_words_lr = pd.DataFrame(bottom_words_lr)

### Figure 4.6: Top 15 Predictive Features per Category (Logistic Regression)
Displays the most influential predictive terms per intent category.

In [ ]:
printmd("#### Figure 4.6: Top 15 Predictive Features per Category (Logistic Regression)")
display(df_top_words_lr)

### 20.2 Logistic Regression Linear Decision Boundary Equations

In [ ]:
def display_decision_boundary_equations_lr(models, feature_names, target_cols):
    printmd("#### Logistic Regression Linear Decision Boundary Equations:")
    for i, label in enumerate(target_cols):
        est = models[i] if isinstance(models, list) else (models.estimators_[i] if hasattr(models, 'estimators_') else models)
        coefs = est.coef_[0]
        intercept = est.intercept_[0]
        top_3_idx = np.argsort(coefs)[-3:][::-1]
        terms = " + ".join([f"({coefs[idx]:.2f} * {feature_names[idx]})" for idx in top_3_idx])
        print(f"[{label:<11}] z = {intercept:.2f} + {terms} + ...")

printmd("#### Figure 4.8: Learned Decision Boundary Equations (Logistic Regression)")
display_decision_boundary_equations_lr(lr_model, feature_names, target_cols)

### 20.3 Sigmoid Activation & Calibrated 1D Probability Curves
### Figure 3.20: Sigmoid Activation Function and Probability Mapping

In [ ]:
printmd("#### Figure 3.20: Sigmoid Activation Function and Probability Mapping")
z = np.linspace(-7, 7, 200)
sigma_z = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4.5))
plt.plot(z, sigma_z, color="#1976d2", linewidth=2.5, label=r"$\sigma(z) = \frac{1}{1 + e^{-z}}$")
plt.axhline(0.5, color="red", linestyle="--", alpha=0.7, label="Classification Threshold (0.5)")
plt.axvline(0.0, color="gray", linestyle=":", alpha=0.7, label="Decision Hyperplane ($z=0$)")
plt.title("Sigmoid Activation Function & Probability Mapping", fontsize=13, fontweight="bold")
plt.xlabel("Logit / Linear Predictor Value ($z$)", fontsize=11, fontweight="bold")
plt.ylabel("Predicted Probability $P(Y=1|x)$", fontsize=11, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower right", fontsize=10)
plt.tight_layout()
plt.show()

### Figure 4.5: Calibrated Sigmoid Probability Curves Across All Six Categories
Plots 1D Sigmoid curves with test post distribution mapping across all six intent categories.

In [ ]:
def plot_sigmoid_probability_curves(models, X_test, target_cols):
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
    sample_indices = np.random.RandomState(42).choice(X_test.shape[0], size=min(300, X_test.shape[0]), replace=False)
    X_sample = X_test[sample_indices]
    y_test_sample = y_test.iloc[sample_indices]
    
    for i, col in enumerate(target_cols):
        est = models[i] if isinstance(models, list) else (models.estimators_[i] if hasattr(models, 'estimators_') else models)
        coefs = est.coef_[0]
        intercept = est.intercept_[0]
        z_scores = X_sample.dot(coefs) + intercept
        probs = 1 / (1 + np.exp(-z_scores))
        z_curve = np.linspace(-6, 6, 200)
        p_curve = 1 / (1 + np.exp(-z_curve))
        
        axes[i].plot(z_curve, p_curve, color='#1976d2', linewidth=2.5, label=r'$\sigma(z)$')
        axes[i].axvline(0, color='gray', linestyle=':', label='Threshold (z=0)')
        axes[i].axhline(0.5, color='gray', linestyle=':')
        
        true_labels = y_test_sample[col].values
        axes[i].scatter(z_scores[true_labels == 1], probs[true_labels == 1], color='#388e3c', alpha=0.6, s=25, label='Actual Positive (1)')
        axes[i].scatter(z_scores[true_labels == 0], probs[true_labels == 0], color='#d32f2f', alpha=0.4, s=25, label='Actual Negative (0)')
        
        axes[i].set_title(f"Sigmoid Curve: {col}", fontsize=12, fontweight='bold')
        axes[i].set_xlabel("Linear Decision Score ($z$)", fontsize=9.5, fontweight='bold')
        axes[i].set_ylabel(r"$P(Y=1|x)$", fontsize=9.5, fontweight='bold')
        axes[i].set_xlim(-6, 6)
        axes[i].set_ylim(-0.05, 1.05)
        axes[i].legend(loc='lower right', fontsize=8)
        axes[i].grid(True, linestyle='--', alpha=0.4)
        
    plt.tight_layout()
    plt.show()

printmd("#### Figure 4.5: Calibrated Sigmoid Probability Curves Across All Six Categories")
plot_sigmoid_probability_curves(lr_model, X_test_tfidf, target_cols)

# Step 21: Logistic Regression Threshold Tuning, Evaluation & Confusion Matrix
### Figure 3.24: Binary Confusion Matrix Template Layout
### Figure 3.25: Mathematical Formulas for Evaluation Metrics

$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2\text{TP}}{2\text{TP} + \text{FP} + \text{FN}}$$

In [ ]:
printmd("#### Figure 3.24: Binary Confusion Matrix Template Layout")
cm_template = np.array([[1, 2], [3, 4]])
labels = [["True Negative (TN)\n(Correctly Absent)", "False Positive (FP)\n(Type I Error / False Alarm)"],
          ["False Negative (FN)\n(Type II Error / Miss)", "True Positive (TP)\n(Correctly Present)"]]

plt.figure(figsize=(6, 4.5))
sns.heatmap(cm_template, annot=labels, fmt="", cmap="Blues", cbar=False,
            xticklabels=["Predicted Negative (0)", "Predicted Positive (1)"],
            yticklabels=["Actual Negative (0)", "Actual Positive (1)"],
            annot_kws={"size": 10, "fontweight": "bold"})
plt.title("Binary Confusion Matrix Template Layout", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Intent Class", fontsize=10, fontweight="bold")
plt.ylabel("Ground Truth Intent Class", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Threshold tuning for optimal F1-score
lr_probs_test = lr_model.predict_proba(X_test_tfidf)
lr_best_thresholds = {}
for i, col in enumerate(target_cols):
    best_thresh, best_f1 = 0.5, 0.0
    for thresh in np.arange(0.2, 0.8, 0.02):
        f1 = f1_score(y_test[col], (lr_probs_test[:, i] >= thresh).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    lr_best_thresholds[col] = best_thresh

y_pred_lr = np.zeros_like(lr_probs_test)
for i, col in enumerate(target_cols):
    y_pred_lr[:, i] = (lr_probs_test[:, i] >= lr_best_thresholds[col]).astype(int)

# Compute Logistic Regression Classification Report
lr_report_data = []
for i, col in enumerate(target_cols):
    acc = accuracy_score(y_test[col], y_pred_lr[:, i])
    prec = precision_score(y_test[col], y_pred_lr[:, i], zero_division=0)
    rec = recall_score(y_test[col], y_pred_lr[:, i], zero_division=0)
    f1 = f1_score(y_test[col], y_pred_lr[:, i], zero_division=0)
    lr_report_data.append({'Intent Category': col, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})

lr_report_df = pd.DataFrame(lr_report_data)
lr_macro = pd.DataFrame([{
    'Intent Category': 'MACRO AVERAGE',
    'Accuracy': lr_report_df['Accuracy'].mean(),
    'Precision': lr_report_df['Precision'].mean(),
    'Recall': lr_report_df['Recall'].mean(),
    'F1-Score': lr_report_df['F1-Score'].mean()
}])
lr_report_df = pd.concat([lr_report_df, lr_macro], ignore_index=True)
printmd("#### Classification Report for Logistic Regression:")
display(lr_report_df.style.format({'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1-Score': '{:.4f}'}))

### Figure 4.2: Logistic Regression Confusion Matrices
Confusion matrix heatmaps for each communicative intent class under Logistic Regression.

In [ ]:
def plot_logistic_regression_confusion_matrices(y_true, y_pred, target_cols):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
    axes = axes.flatten()
    for i, col in enumerate(target_cols):
        y_true_col = y_true[col].values if hasattr(y_true, col) else y_true[:, i]
        y_pred_col = y_pred[:, i] if isinstance(y_pred, np.ndarray) else y_pred[col].values
        cm = confusion_matrix(y_true_col, y_pred_col)
        sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[i], cbar=False,
                    annot_kws={"size": 11, "fontweight": "bold"},
                    xticklabels=['Absent (0)', 'Present (1)'],
                    yticklabels=['Absent (0)', 'Present (1)'])
        axes[i].set_title(f"Logistic Regression: {col}", fontsize=12, fontweight='bold')
        axes[i].set_xlabel("Predicted Label", fontsize=10, fontweight='bold')
        axes[i].set_ylabel("True Label", fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

printmd("#### Figure 4.2: Logistic Regression Confusion Matrices")
plot_logistic_regression_confusion_matrices(y_test, y_pred_lr, target_cols)

# Step 22: Train One-vs-Rest Linear Support Vector Classifier (Linear SVC) Model

### Figure 3.21: Linear Support Vector Classifier (Linear SVC) Optimization Objective
$$\min_{\mathbf{w}_k, b_k} \frac{1}{2} \|\mathbf{w}_k\|^2 + C \sum_{i=1}^N \max\left(0, 1 - y_k^{(i)}(\mathbf{w}_k^T \mathbf{x}^{(i)} + b_k)\right)$$

### Figure 3.22: Hinge Loss Formula for Linear Support Vector Machines
$$\mathcal{L}_{\text{Hinge}}(y, z) = \max(0, 1 - y \cdot z)$$

In [ ]:
from sklearn.svm import LinearSVC

svc_model = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=2000, random_state=42))
svc_model.fit(X_train_tfidf, y_train)
svc_models = svc_model.estimators_

# Persist Linear SVC model artifact
joblib_dump(svc_model, "linear_svc_model.joblib")
printmd("**Linear SVC Model:** Successfully trained and saved to `linear_svc_model.joblib`.")

# Step 23: Linear SVC Feature Interpretation & Prediction Rule
### 23.1 Coefficient Evaluation: Top & Bottom Predictive Features per Intent Class (Linear SVC)

In [ ]:
top_words_svc = {}
bottom_words_svc = {}

for i, label in enumerate(target_cols):
    coefs = svc_model.estimators_[i].coef_[0]
    top_indices = np.argsort(coefs)[-15:][::-1]
    bottom_indices = np.argsort(coefs)[:15]
    top_words_svc[label] = [f"{feature_names[idx]} ({coefs[idx]:.2f})" for idx in top_indices]
    bottom_words_svc[label] = [f"{feature_names[idx]} ({coefs[idx]:.2f})" for idx in bottom_indices]

df_top_words_svc = pd.DataFrame(top_words_svc)
df_bottom_words_svc = pd.DataFrame(bottom_words_svc)

### Figure 4.7: Top 15 Predictive Features per Category (Linear SVC)
Displays the most influential predictive terms per intent category under Linear SVC.

In [ ]:
printmd("#### Figure 4.7: Top 15 Predictive Features per Category (Linear SVC)")
display(df_top_words_svc)

### 23.2 Linear Decision Boundary Equations for Linear SVC

In [ ]:
def display_decision_boundary_equations_svc(models, feature_names, target_cols):
    printmd("#### Linear SVC Linear Decision Boundary Equations:")
    for i, label in enumerate(target_cols):
        est = models[i] if isinstance(models, list) else (models.estimators_[i] if hasattr(models, 'estimators_') else models)
        coefs = est.coef_[0]
        intercept = est.intercept_[0]
        top_3_idx = np.argsort(coefs)[-3:][::-1]
        terms = " + ".join([f"({coefs[idx]:.2f} * {feature_names[idx]})" for idx in top_3_idx])
        print(f"[{label:<11}] z = {intercept:.2f} + {terms} + ...")

printmd("#### Figure 4.9: Learned Decision Boundary Equations (Linear SVC)")
display_decision_boundary_equations_svc(svc_model, feature_names, target_cols)

### 23.3 Linear SVC Margin and Hyperplane (Conceptual 2D)
### Figure 3.23: Linear SVC Margin and Hyperplane (Conceptual 2D)

In [ ]:
printmd("#### Figure 3.23: Linear SVC Margin and Hyperplane (Conceptual 2D)")
np.random.seed(42)
X_neg = np.random.randn(25, 2) + np.array([-2, -2])
X_pos = np.random.randn(25, 2) + np.array([2, 2])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_neg[:, 0], X_neg[:, 1], color="#d32f2f", edgecolors="black", s=60, label="Negative Class ($y = -1$)")
ax.scatter(X_pos[:, 0], X_pos[:, 1], color="#388e3c", edgecolors="black", s=60, label="Positive Class ($y = +1$)")

x_vals = np.linspace(-4, 4, 100)
ax.plot(x_vals, -x_vals, 'k-', linewidth=2, label="Optimal Hyperplane ($\\mathbf{w}^T\\mathbf{x} + b = 0$)")
ax.plot(x_vals, -x_vals + 1.4, 'b--', alpha=0.7, label=r"Positive Margin ($\\mathbf{w}^T\\mathbf{x} + b = +1$)")
ax.plot(x_vals, -x_vals - 1.4, 'r--', alpha=0.7, label=r"Negative Margin ($\\mathbf{w}^T\\mathbf{x} + b = -1$)")
ax.fill_between(x_vals, -x_vals - 1.4, -x_vals + 1.4, color="gray", alpha=0.15, label="Maximum Margin ($\\frac{2}{\\\|\\mathbf{w}\\\|}$)")

ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_title("Linear Support Vector Classifier Maximum Margin Hyperplane", fontsize=13, fontweight="bold")
ax.set_xlabel("Feature Dimension $x_1$", fontsize=11, fontweight="bold")
ax.set_ylabel("Feature Dimension $x_2$", fontsize=11, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Step 24: Linear SVC Evaluation & Confusion Matrix

In [ ]:
y_pred_svc = svc_model.predict(X_test_tfidf)

# Compute Linear SVC Classification Report
svc_report_data = []
for i, col in enumerate(target_cols):
    acc = accuracy_score(y_test[col], y_pred_svc[:, i])
    prec = precision_score(y_test[col], y_pred_svc[:, i], zero_division=0)
    rec = recall_score(y_test[col], y_pred_svc[:, i], zero_division=0)
    f1 = f1_score(y_test[col], y_pred_svc[:, i], zero_division=0)
    svc_report_data.append({'Intent Category': col, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})

svc_report_df = pd.DataFrame(svc_report_data)
svc_macro = pd.DataFrame([{
    'Intent Category': 'MACRO AVERAGE',
    'Accuracy': svc_report_df['Accuracy'].mean(),
    'Precision': svc_report_df['Precision'].mean(),
    'Recall': svc_report_df['Recall'].mean(),
    'F1-Score': svc_report_df['F1-Score'].mean()
}])
svc_report_df = pd.concat([svc_report_df, svc_macro], ignore_index=True)
printmd("#### Classification Report for Linear SVC:")
display(svc_report_df.style.format({'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1-Score': '{:.4f}'}))

### Figure 4.3: Linear SVC Confusion Matrices
Confusion matrix heatmaps for each communicative intent class under Linear SVC.

In [ ]:
def plot_linear_svc_confusion_matrices(y_true, y_pred, target_cols):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
    axes = axes.flatten()
    for i, col in enumerate(target_cols):
        y_true_col = y_true[col].values if hasattr(y_true, col) else y_true[:, i]
        y_pred_col = y_pred[:, i] if isinstance(y_pred, np.ndarray) else y_pred[col].values
        cm = confusion_matrix(y_true_col, y_pred_col)
        sns.heatmap(cm, annot=True, fmt=',d', cmap='Purples', ax=axes[i], cbar=False,
                    annot_kws={"size": 11, "fontweight": "bold"},
                    xticklabels=['Absent (0)', 'Present (1)'],
                    yticklabels=['Absent (0)', 'Present (1)'])
        axes[i].set_title(f"Linear SVC: {col}", fontsize=12, fontweight='bold')
        axes[i].set_xlabel("Predicted Label", fontsize=10, fontweight='bold')
        axes[i].set_ylabel("True Label", fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

printmd("#### Figure 4.3: Linear SVC Confusion Matrices")
plot_linear_svc_confusion_matrices(y_test, y_pred_svc, target_cols)

# Step 25: Side-by-Side Model Comparison (Logistic Regression vs Linear SVC)
### Figure 4.1: Overall Performance Comparison Table
### Figure 4.4: Comparative Bar Charts Across Communicative Categories

In [ ]:
# Construct side-by-side comparison DataFrame
comparison_rows = []
for cat in target_cols + ['MACRO AVERAGE']:
    lr_row = lr_report_df[lr_report_df['Intent Category'] == cat].iloc[0]
    svc_row = svc_report_df[svc_report_df['Intent Category'] == cat].iloc[0]
    comparison_rows.append({
        'Intent Category': cat,
        'LR_Accuracy': lr_row['Accuracy'],
        'SVC_Accuracy': svc_row['Accuracy'],
        'LR_Precision': lr_row['Precision'],
        'SVC_Precision': svc_row['Precision'],
        'LR_Recall': lr_row['Recall'],
        'SVC_Recall': svc_row['Recall'],
        'LR_F1-Score': lr_row['F1-Score'],
        'SVC_F1-Score': svc_row['F1-Score']
    })

df_model_comparison = pd.DataFrame(comparison_rows).set_index('Intent Category')

printmd("#### Figure 4.1: Overall Performance Comparison Table")
display(df_model_comparison.style.format('{:.3f}'))

# Plot comparison bar charts
def plot_model_comparison_bars(comparison_df):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    categories = target_cols + ['MACRO AVERAGE']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    x = np.arange(len(categories))
    width = 0.35
    
    for idx, metric in enumerate(metrics):
        lr_vals = [comparison_df.loc[cat, f'LR_{metric}'] for cat in categories]
        svc_vals = [comparison_df.loc[cat, f'SVC_{metric}'] for cat in categories]
        
        rects1 = axes[idx].bar(x - width/2, lr_vals, width, label='Logistic Regression', color='#1976d2', edgecolor='black')
        rects2 = axes[idx].bar(x + width/2, svc_vals, width, label='Linear SVC', color='#7b1fa2', edgecolor='black')
        
        axes[idx].set_title(f"Model Comparison: {metric}", fontsize=12, fontweight='bold')
        axes[idx].set_xticks(x)
        axes[idx].set_xticklabels(categories, rotation=20, ha='right', fontsize=9.5)
        axes[idx].set_ylabel(metric, fontsize=10, fontweight='bold')
        axes[idx].set_ylim(0, 1.15)
        axes[idx].legend(loc='upper right', fontsize=9.5)
        axes[idx].grid(True, linestyle='--', alpha=0.4, axis='y')
        
        for r in rects1:
            h = r.get_height()
            axes[idx].text(r.get_x() + r.get_width()/2., h + 0.015, f"{h:.3f}", ha='center', va='bottom', fontsize=8, rotation=90)
        for r in rects2:
            h = r.get_height()
            axes[idx].text(r.get_x() + r.get_width()/2., h + 0.015, f"{h:.3f}", ha='center', va='bottom', fontsize=8, rotation=90)
            
    plt.tight_layout()
    plt.show()

printmd("#### Figure 4.4: Comparative Bar Charts Across Communicative Categories")
plot_model_comparison_bars(df_model_comparison)

# Step 26: End-to-End Raw Text Preprocessing Pipeline & Model Inference Demo
Packages the full 16-step preprocessing sequence into a single unified callable function `preprocess_pipeline()` and loads saved artifacts.

In [ ]:
def preprocess_pipeline(raw_text):
    if not isinstance(raw_text, str) or not raw_text.strip():
        return ""
    
    # Step 1: Remove Quotes, BBCode & Markup
    text = remove_forum_markup(raw_text)
    
    # Step 2: Entity Masking
    text = mask_entities(text)
    
    # Step 3: Normalize Elongations & Laughter
    text = normalize_elongation_and_laughter(text)
    
    # Step 4: Convert Emojis
    text = convert_emojis(text)
    
    # Step 5: Case & Spacing Normalization
    text = normalize_spacing_and_case(text)
    
    # Step 6: Slang Normalization
    text = normalize_slang_hybrid(text)
    
    # Step 7: Expand Contractions
    text = fix_contractions(text)
    
    # Step 8: Remove All Punctuations
    text = remove_punctuations(text)
    
    # Step 9: Split Alphanumeric & Remove Standalone Numbers
    text = split_and_remove_numbers(text)
    
    # Step 10: Remove Non-Latin Characters
    text = remove_non_latin(text)
    
    # Step 11: Word Tokenization
    tokens = tokenize_words(text)
    
    # Step 12: Language Tagging
    tagged = tag_tokens(tokens)
    
    # Step 13: POS Tagging for English Tokens
    pos_tagged = pos_tag_english_tokens(tagged)
    
    # Step 14: Lemmatization & Stemming
    morph_tokens = lemmatize_and_stem(pos_tagged)
    
    # Step 15: Stopword Removal
    filtered = remove_stopwords(morph_tokens)
    
    return ' '.join([t[0] for t in filtered])

# Load serialized artifacts from disk
loaded_tfidf = joblib_load("tfidf_vectorizer.joblib")
loaded_lr = joblib_load("logistic_regression_model.joblib")
loaded_svc = joblib_load("linear_svc_model.joblib")
printmd("**Artifacts Status:** All serialized models (`TF-IDF`, `Logistic Regression`, `Linear SVC`) loaded successfully from disk.")

# Step 27: Real-Time Prediction on Unseen Posts (Showing Both Models)
Execute real-time multi-label classification on brand-new unseen forum posts, displaying predictions simultaneously from **both Logistic Regression and Linear SVC**.

In [ ]:
def format_ranked_badges(ranked_list, is_probability=True):
    if not ranked_list:
        return '<span style="color:#777; font-style:italic; font-size:12px;">Neutral / None</span>'
    
    badges = []
    for idx, (intent, score) in enumerate(ranked_list):
        score_text = f"{score * 100:.1f}%" if is_probability else f"score: {score:+.2f}"
        if idx == 0:
            badges.append(
                f'<span style="background-color:#2e7d32; color:#ffffff; padding: 3px 9px; '
                f'border-radius: 4px; font-weight: bold; font-size: 12px; margin-right: 4px;">'
                f'★ Best: {intent} ({score_text})</span>'
            )
        else:
            badges.append(
                f'<span style="background-color:#e0e0e0; color:#333333; padding: 3px 8px; '
                f'border-radius: 4px; font-weight: 500; font-size: 12px; margin-right: 4px;">'
                f'{intent} ({score_text})</span>'
            )
    return ''.join(badges)

def predict_unseen(raw_text):
    cleaned_input = preprocess_pipeline(raw_text)
    if not cleaned_input.strip():
        return {
            "raw_text": raw_text,
            "cleaned_text": cleaned_input,
            "lr_ranked": [],
            "svc_ranked": []
        }
    
    vectorized_input = loaded_tfidf.transform([cleaned_input])
    
    # Logistic Regression
    lr_probs = loaded_lr.predict_proba(vectorized_input)[0]
    lr_positive = [(target_cols[i], lr_probs[i]) for i in range(len(target_cols)) if lr_probs[i] >= lr_best_thresholds[target_cols[i]]]
    lr_ranked = sorted(lr_positive, key=lambda x: x[1], reverse=True)
    
    # Linear SVC
    svc_scores = loaded_svc.decision_function(vectorized_input)[0]
    svc_preds = loaded_svc.predict(vectorized_input)[0]
    svc_positive = [(target_cols[i], svc_scores[i]) for i, val in enumerate(svc_preds) if val == 1]
    svc_ranked = sorted(svc_positive, key=lambda x: x[1], reverse=True)
    
    return {
        "raw_text": raw_text,
        "cleaned_text": cleaned_input,
        "lr_ranked": lr_ranked,
        "svc_ranked": svc_ranked
    }

# Test sample comment
test_sample = "can you check whether i did it right or not why is it failing"
res = predict_unseen(test_sample)
printmd(f"**Test Prediction Demo on Sample Input:** `{test_sample}`")
display(HTML(f'''
<div style="border: 1px solid #e0e0e0; border-radius: 6px; padding: 12px; margin-top: 10px; background-color: #fafafa;">
    <div style="margin-bottom: 6px;"><strong>Raw Input:</strong> <code style="font-size:12px;">{html.escape(res['raw_text'])}</code></div>
    <div style="margin-bottom: 8px;"><strong>Cleaned Input:</strong> <code style="font-size:12px;">{html.escape(res['cleaned_text'])}</code></div>
    <div style="margin-bottom: 6px;"><strong>Logistic Regression:</strong> {format_ranked_badges(res['lr_ranked'], is_probability=True)}</div>
    <div style="margin-bottom: 6px;"><strong>Linear SVC:</strong> {format_ranked_badges(res['svc_ranked'], is_probability=False)}</div>
</div>
'''))